In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
from PIL import Image
import numpy as np
from sklearn.preprocessing import LabelEncoder
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import io
from helper import *
import warnings

from sklearn.model_selection import GridSearchCV

c:\Users\Fran\miniconda3\envs\entorno_skin\Lib\site-packages\albumentations\__init__.py:28: UserWarning: A new version of Albumentations is available: '2.0.8' (you have '2.0.7'). Upgrade using: pip install -U albumentations. To disable automatic update checks, set the environment variable NO_ALBUMENTATIONS_UPDATE to 1.
  check_for_updates()


In [2]:
warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
import mlflow
import mlflow.pytorch

In [4]:
from torch.utils.tensorboard import SummaryWriter
import torchvision.utils as vutils

In [5]:
import torch_directml

In [6]:
mlflow.set_experiment("Clasificador_Imagenes")

2026/06/01 14:45:31 INFO mlflow.tracking.fluent: Experiment with name 'Clasificador_Imagenes' does not exist. Creating a new experiment.


<Experiment: artifact_location='file:///c:/Users/Fran/Desktop/Facultad/Redes%20Neuronales/Tp1/skin-dataset-classification/mlruns/758850062710091186', creation_time=1780335931522, experiment_id='758850062710091186', last_update_time=1780335931522, lifecycle_stage='active', name='Clasificador_Imagenes', tags={}>

In [7]:
def log_classification_report(model, loader, writer, device, classes, step, prefix="val", log_tensorboard=True, log_mlflow=False):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    # Cálculo del reporte de texto
    cls_report = classification_report(all_labels, all_preds, target_names=classes, zero_division=0)

    if log_tensorboard:
        writer.add_text(f"{prefix}/classification_report", f"<pre>{cls_report}</pre>", step)

    # REPORTE MAESTRO (Solo Mejor Época): Genera los artefactos para MLflow
    if log_mlflow:
        # Confusion matrix
        cm = confusion_matrix(all_labels, all_preds)
        fig_cm, ax = plt.subplots(figsize=(6, 6))
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=classes)
        disp.plot(ax=ax, cmap='Blues', xticks_rotation=45)

        #Forzar la alineación del texto a la derecha para que el final de la palabra coincida con la marca del eje
        plt.setp(ax.get_xticklabels(), ha="right", rotation_mode="anchor")

        ax.set_title(f'{prefix.title()} - Confusion Matrix (Best Epoch)')

        # Ajustar automáticamente los márgenes para que no se corte nada
        plt.tight_layout()

        # Guardar localmente y subir a MLflow
        fig_path = f"confusion_matrix_{prefix}_epoch_{step}.png"
        fig_cm.savefig(fig_path)
        mlflow.log_artifact(fig_path)
        os.remove(fig_path)
        plt.close(fig_cm) # Importante para liberar la RAM de los gráficos

    
        # Reporte de texto definitivo para MLflow
        with open(f"classification_report_{prefix}_epoch_{step}.txt", "w") as f:
            f.write(cls_report)
        mlflow.log_artifact(f.name)
        os.remove(f.name)


In [8]:
# Entrenamiento y validación
def evaluate(model, loader, writer, device, classes, criterion, epoch=None, prefix="val"):
    # Activamos SOLO TensorBoard para ver la evolución por época
    if epoch is not None:
        log_classification_report(
            model, loader, writer, device, classes, step=epoch, prefix=prefix, 
            log_tensorboard=True, log_mlflow=False
        )
    model.eval()
    correct, total, loss_sum = 0, 0, 0.0

    with torch.no_grad():
        for i, (images, labels) in enumerate(loader):
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            outputs = model(images)
            loss = criterion(outputs, labels)
            _, preds = torch.max(outputs, 1)

            loss_sum += loss.item()
            correct += (preds == labels).sum().item()
            total += labels.size(0)

            # Loguear imágenes del primer batch
            if i == 0 and epoch is not None:
                img_grid = vutils.make_grid(images[:8].cpu(), normalize=True)
                writer.add_image(f"{prefix}/images", img_grid, global_step=epoch)

    acc = 100.0 * correct / total
    avg_loss = loss_sum / len(loader)

    if epoch is not None:
        writer.add_scalar(f"{prefix}/loss", avg_loss, epoch)
        writer.add_scalar(f"{prefix}/accuracy", acc, epoch)

    return avg_loss, acc

In [9]:
class CustomImageDataset(Dataset):
    def __init__(self, root_dir=None, transform=None, image_paths=None, labels=None, label_encoder=None, images_in_ram=None):
        self.transform = transform
        self.images_in_ram = images_in_ram
        
        # Si se tienen las imágenes en RAM o las rutas listas con sus etiquetas, se evita el HDD
        if (images_in_ram is not None or image_paths is not None) and labels is not None:
            self.image_paths = image_paths  
            self.labels = labels
            self.label_encoder = label_encoder
        else:
            # Solo se ejecuta en la pre-carga única del HDD
            self.root_dir = root_dir
            self.image_paths = []
            self.labels = []

            class_names = sorted(os.listdir(root_dir))
            self.class_to_idx = {cls: idx for idx, cls in enumerate(class_names)}

            for cls in class_names:
                cls_dir = os.path.join(root_dir, cls)
                for fname in os.listdir(cls_dir):
                    if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                        self.image_paths.append(os.path.join(cls_dir, fname))
                        self.labels.append(cls)

            self.label_encoder = LabelEncoder()
            self.labels = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        # Si no hay rutas porque usamos la RAM, medimos el tamaño usando la lista de imágenes o etiquetas
        if self.images_in_ram is not None:
            return len(self.images_in_ram)
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Si las imágenes ya están cargadas en RAM, las tomamos de la lista 
        if self.images_in_ram is not None:
            image = self.images_in_ram[idx]
        else:
            #Comportamiento original de leer el HDD 
            image = np.array(Image.open(self.image_paths[idx]).convert("RGB"))

        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented["image"]

        return image, label

In [10]:
def get_transforms(strategy, size):
    if strategy == "direct_resize":
        # Compresión por Resize
        return [A.Resize(height=size, width=size)]
        
    elif strategy == "padding_preserve":
        # Compresión preservando la relación de aspecto original
        return [
            A.LongestMaxSize(max_size=size),
            A.PadIfNeeded(min_height=size, min_width=size, border_mode=0)
        ]
    else:
        raise ValueError(f"Estrategia de redimensionamiento desconocida: {strategy}")

In [11]:
class MLPClassifier(nn.Module):
    def __init__(self, input_size=64*64*3, hidden_layers=[512, 128], num_classes=9, dropout_rate=0.0):
        super().__init__()
        
        # 1. Lista inicial
        layers = []
        
        # 2. Construcción dinámica de las capas ocultas
        current_input_dim = input_size
        
        for hidden_dim in hidden_layers:
            layers.append(nn.Linear(current_input_dim, hidden_dim)) # Capa lineal
            layers.append(nn.ReLU())                                # Activación
            if dropout_rate > 0.0:
                layers.append(nn.Dropout(dropout_rate))             # Dropout condicional
            
            # El tamaño de salida de esta capa será la entrada de la siguiente
            current_input_dim = hidden_dim
            
        # 3. Capa final de salida (clasificación)
        layers.append(nn.Linear(current_input_dim, num_classes))
        
        # 4. Conversión de la lista de Python en el contenedor secuencial de PyTorch
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        # Ajuste para GPU: Forzar un aplanado contiguo en la VRAM
        x = torch.flatten(x, start_dim=1)
        return self.model(x)

In [12]:
# Paths
train_dir = "data/Split_smol/train"
val_dir = "data/Split_smol/val/"

In [13]:
# Crear directorio de logs de tensorboard
log_dir = "runs/experimento_skin"
writer = SummaryWriter(log_dir=log_dir)

In [14]:
hparams_space= {
    "model": ("MLPClassifier"),
    "hidden_layers": [
        [128],              # Arquitectura 1: Superficial
        [512, 128]          # Arquitectura 2: Media
    ],
    "input_size":  [64, 128],
    "batch_size": [16, 64],
    "lr": [1e-2, 1e-3, 1e-4],
    "epochs": 100,
    "optimizer":  ["RMSprop", "SGD"],   
    "HFlip": [0.5],
    "VFlip": [0.5],
    "RBContrast": [0.0, 0.5],
    "loss_fn": "CrossEntropyLoss",
    "train_dir": train_dir,
    "val_dir": val_dir,
    "es_patience": 5,
    "dropout": [0.0, 0.2],
    "weight_decay": [0.0, 1e-4, 1e-3],
    "Rotate": [0.0, 0.5],
    "Shift": [0.0, 0.3],
    "resize_strategy": ["direct_resize", "padding_preserve"]
}

In [15]:
# ==========================================
# 1. PRE-CARGA DE DATOS (Fuera de los bucles)
# ==========================================
print("Escaneando directorios de imágenes por única vez...")
# Cargador temporal sin transformaciones para extraer las imágenes crudas
temp_train_dataset = CustomImageDataset(root_dir=train_dir, transform=None)
temp_val_dataset = CustomImageDataset(root_dir=val_dir, transform=None)

# Forzamos la lectura secuencial de todo el disco rígido hacia listas en RAM
train_images_ram = [temp_train_dataset[i][0] for i in range(len(temp_train_dataset))]
train_labels = temp_train_dataset.labels
train_le = temp_train_dataset.label_encoder

val_images_ram = [temp_val_dataset[i][0] for i in range(len(temp_val_dataset))]
val_labels = temp_val_dataset.labels
val_le = temp_val_dataset.label_encoder

# Configuración de hardware
device = torch_directml.device()
num_classes = len(train_le.classes_)

modelnbr = 0

# ====================================================
# HIPERPARÁMETROS QUE DEFINEN EL DATASET Y EL LOADER
# ====================================================
for input_size in hparams_space["input_size"]:
    for resize_strategy in hparams_space["resize_strategy"]:
        for batch_size in hparams_space["batch_size"]:
            for HFlip in hparams_space["HFlip"]:
                for VFlip in hparams_space["VFlip"]:
                    for Rotate in hparams_space["Rotate"]:
                        for Shift in hparams_space["Shift"]:
                            for RBContrast in hparams_space["RBContrast"]:
                                
                                # CONFIGURACIÓN DE PROCESAMIENTO 
                                train_transform = A.Compose([
                                    *get_transforms(resize_strategy, input_size),
                                    A.HorizontalFlip(p=HFlip),
                                    A.VerticalFlip(p=VFlip),
                                    A.Rotate(limit=90 if Rotate > 0 else 0, p=Rotate),
                                    A.ShiftScaleRotate(shift_limit=0.1 if Shift > 0 else 0.0, scale_limit=0.0, rotate_limit=0, p=Shift),
                                    A.RandomBrightnessContrast(p=RBContrast),
                                    A.Normalize(),
                                    ToTensorV2()
                                ])
                                
                                val_test_transform = A.Compose([
                                    *get_transforms(resize_strategy, input_size),
                                    A.Normalize(),
                                    ToTensorV2()
                                ])
                                
                                # ===============================================
                                # Modifico el Dataset para que lea de la RAM
                                # ===============================================
                                
                                train_dataset = CustomImageDataset(
                                    transform=train_transform, 
                                    images_in_ram=train_images_ram, # <-- Lee desde la memoria volatíl
                                    labels=train_labels, 
                                    label_encoder=train_le
                                )
                                val_dataset = CustomImageDataset(
                                    transform=val_test_transform, 
                                    images_in_ram=val_images_ram, 
                                    labels=val_labels, 
                                    label_encoder=val_le
                                )
                                
                                # Cargadores optimizados con Pin Memory 
                                train_loader = DataLoader(
                                    train_dataset, 
                                    batch_size=batch_size, 
                                    shuffle=True, 
                                    num_workers=0, 
                                    pin_memory=True 
                                )
                                val_loader = DataLoader(
                                    val_dataset, 
                                    batch_size=batch_size, 
                                    num_workers=0, 
                                    pin_memory=True
                                )
                                
                                # =================================================================
                                # HIPERPARÁMETROS MATEMÁTICOS (Modifican el Modelo, no los Datos)
                                # =================================================================
                                for hidden_layers in hparams_space["hidden_layers"]:
                                    for lr in hparams_space["lr"]:
                                        for optimizer_name in hparams_space["optimizer"]:
                                            for dropout in hparams_space["dropout"]:
                                                for weight_decay in hparams_space["weight_decay"]:
                                                    
                                                    # Filtro de Random Search al 10%
                                                    if np.random.rand() < 0.1:
                                                        print(f"Entrenando modelo número: {modelnbr}...", end="\r")
                                                        modelnbr += 1
                                                        
                                                        # Armando hparams para MLflow...
                                                        hparams = {
                                                            "model": "MLPClassifier", 
                                                            "hidden_layers": hidden_layers, 
                                                            "input_size": input_size,
                                                            "batch_size": batch_size, 
                                                            "lr": lr, 
                                                            "epochs": hparams_space["epochs"],
                                                            "optimizer": optimizer_name, 
                                                            "HFlip": HFlip, 
                                                            "VFlip": VFlip, 
                                                            "Rotate": Rotate,
                                                            "Shift": Shift, 
                                                            "RBContrast": RBContrast, 
                                                            "loss_fn": "CrossEntropyLoss",
                                                            "train_dir": train_dir, 
                                                            "val_dir": val_dir, 
                                                            "es_patience": hparams_space["es_patience"],
                                                            "dropout": dropout, 
                                                            "weight_decay": weight_decay, 
                                                            "resize_strategy": resize_strategy
                                                        }
                                                        
                                                        # Inicialización directa en GPU
                                                        model = MLPClassifier(
                                                            input_size=input_size**2 * 3, 
                                                            hidden_layers=hidden_layers, 
                                                            num_classes=num_classes, 
                                                            dropout_rate=dropout
                                                        ).to(device)
                                                        
                                                        criterion = nn.CrossEntropyLoss()
                                                        optimizer = optim.RMSprop(model.parameters(), lr=lr, weight_decay=weight_decay) if optimizer_name == "RMSprop" else optim.SGD(model.parameters(), lr=lr, weight_decay=weight_decay)
                                                        
                                                         # tracking MLflow
                                                        with mlflow.start_run():
                                                            mlflow.log_params(hparams)
                                                            best_val_acc = 0.0
                                                            best_val_loss = 0.0
                                                            best_train_acc = 0.0
                                                            best_train_loss = 0.0
                                                            gap_accuracy = 0.0
                                                            best_epoch = 0
                                                            
                                                            for epoch in range(hparams["epochs"]):
                                                                model.train()
                                                                running_loss = 0.0
                                                                correct, total = 0, 0
                                                                
                                                                for images, labels in train_loader:
                                                                    # non_blocking=True elimina la espera de sincronización de la CPU
                                                                    images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                                                                    
                                                                    optimizer.zero_grad()
                                                                    outputs = model(images)
                                                                    loss = criterion(outputs, labels)
                                                                    loss.backward()
                                                                    optimizer.step()
                                                                    
                                                                    running_loss += loss.item()
                                                                    _, preds = torch.max(outputs, 1)
                                                                    correct += (preds == labels).sum().item()
                                                                    total += labels.size(0)
                                                                
                                                                train_loss = running_loss / len(train_loader)
                                                                train_acc = 100.0 * correct / total
                                                                
                                                                # Evaluación usando tus funciones originales intactas
                                                                val_loss, val_acc = evaluate(model, val_loader, writer, device, train_le.classes_, criterion, epoch=epoch, prefix="val")
                                                                
                                                                # Logs intermedios en TensorBoard y MLflow
                                                                writer.add_scalar("train/loss", train_loss, epoch)
                                                                writer.add_scalar("train/accuracy", train_acc, epoch)
                                                                
                                                                mlflow.log_metrics({
                                                                    "train_loss": train_loss,
                                                                    "train_accuracy": train_acc,
                                                                    "val_loss": val_loss,
                                                                    "val_accuracy": val_acc
                                                                }, step=epoch)
                                                                
                                                                # Lógica de guardado del mejor modelo por Run
                                                                if val_acc > best_val_acc:
                                                                    best_val_acc = val_acc
                                                                    best_val_loss = val_loss
                                                                    best_train_acc = train_acc
                                                                    best_train_loss = train_loss
                                                                    gap_accuracy = best_train_acc - best_val_acc
                                                                    best_epoch = epoch
                                                                    
                                                                    if os.path.exists("best_mlp_model.pth"):
                                                                        os.remove("best_mlp_model.pth")
                                                                    torch.save(model.state_dict(), "best_mlp_model.pth")
                                                                
                                                                # Early Stopping
                                                                if epoch > best_epoch + hparams["es_patience"]:
                                                                    break
                                                            
                                                            # Fin del entrenamiento de este modelo: Recuperar pesos de la mejor época
                                                            if os.path.exists("best_mlp_model.pth"):
                                                                model.load_state_dict(torch.load("best_mlp_model.pth"))
                                                            
                                                            # Reporte maestro final a MLflow de fin de ciclo
                                                            log_classification_report(
                                                                model, val_loader, writer, device, 
                                                                train_le.classes_, 
                                                                step=best_epoch, prefix="val", 
                                                                log_tensorboard=False, log_mlflow=True
                                                            )
                                                            
                                                            if os.path.exists("best_mlp_model.pth"):
                                                                os.remove("best_mlp_model.pth")
                                                            
                                                            mlflow.log_metrics({
                                                                "train_loss": best_train_loss,
                                                                "train_accuracy": best_train_acc,
                                                                "val_loss": best_val_loss,
                                                                "val_accuracy": best_val_acc,
                                                                "best_epoch": best_epoch,
                                                                "overfitting_gap": gap_accuracy
                                                            }, step=epoch+1)

Escaneando directorios de imágenes por única vez...


c:\Users\Fran\miniconda3\envs\entorno_skin\Lib\site-packages\albumentations\core\validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


c:\Users\Fran\miniconda3\envs\entorno_skin\Lib\site-packages\albumentations\core\validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


c:\Users\Fran\miniconda3\envs\entorno_skin\Lib\site-packages\albumentations\core\validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


c:\Users\Fran\miniconda3\envs\entorno_skin\Lib\site-packages\albumentations\core\validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


...

c:\Users\Fran\miniconda3\envs\entorno_skin\Lib\site-packages\albumentations\core\validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


c:\Users\Fran\miniconda3\envs\entorno_skin\Lib\site-packages\albumentations\core\validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


c:\Users\Fran\miniconda3\envs\entorno_skin\Lib\site-packages\albumentations\core\validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


c:\Users\Fran\miniconda3\envs\entorno_skin\Lib\site-packages\albumentations\core\validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


c:\Users\Fran\miniconda3\envs\entorno_skin\Lib\site-packages\albumentations\core\validation.py:111: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [16]:
!mlflow ui

^C


In [ ]:
#http://localhost:5000